[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/03_ONNX_Architecture_and_Internals/03_ONNX_IR_Specification/ONNX_IR_Specification_Deep_Dive.ipynb)

# 3.3 ONNX IR Specification — Deep Dive

The ONNX **Intermediate Representation (IR)** is a Protocol Buffer–based specification that defines the complete wire format for neural network models. This section dissects every layer of the protobuf hierarchy from `ModelProto` down to individual tensor bytes.

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [What is the ONNX IR?](#section-1) | Purpose, versioning, and relationship to protobuf |
| 2 | [The Protobuf Hierarchy](#section-2) | Full decomposition from Model to bytes |
| 3 | [ModelProto](#section-3) | Top-level container fields |
| 4 | [GraphProto](#section-4) | The computation graph container |
| 5 | [NodeProto and AttributeProto](#section-5) | Operators and their parameters |
| 6 | [TensorProto and ValueInfoProto](#section-6) | Data storage and type descriptors |
| 7 | [OpSetImportProto and IR Versioning](#section-7) | Version compatibility system |
| 8 | [Serialization and Binary Format](#section-8) | Wire format details |
| 9 | [Inspecting Real Models](#section-9) | Code walkthrough |
| 10 | [Key Takeaways & Interview Questions](#section-10) | Summary |

### Prerequisites

- Completed **3.1** and **3.2** (Computation Graphs, Nodes/Edges/Tensors)
- Basic understanding of Protocol Buffers

<a id='section-1'></a>
## Section 1: What is the ONNX IR?

### Definition

The ONNX IR is a **formal specification** that defines:

1. **Syntax** — What protobuf messages exist and how they nest
2. **Semantics** — What each field means and how a runtime should interpret it
3. **Versioning** — How the specification evolves without breaking backward compatibility

### IR Version History

| IR Version | Key Changes |
|:----------:|-------------|
| 1–3 | Initial versions |
| 4 | Added `metadata_props` |
| 5 | Subgraph support (`If`, `Loop`) |
| 6 | Training support additions |
| 7 | Function support |
| 8 | Type/shape in outer graph context for subgraphs |
| 9 | Sparse tensor support, optional types |
| 10 | Latest: improved type system |

### Relationship to Protocol Buffers

ONNX uses Protocol Buffers (protobuf) as its serialization format:

```
Conceptual Model           Protobuf Schema           Binary File
┌──────────────┐          ┌───────────────┐         ┌──────────┐
│ Neural Network│  ──▶    │  .proto defs   │  ──▶   │ .onnx    │
│ Architecture  │ define  │  (onnx.proto)  │ encode │ (bytes)  │
└──────────────┘          └───────────────┘         └──────────┘
```

### Why Protobuf?

| Feature | Benefit for ONNX |
|---------|------------------|
| **Language-neutral** | Generate parsers for C++, Python, Java, etc. |
| **Backward compatible** | New fields don't break old readers |
| **Compact binary** | Models are smaller than JSON/XML |
| **Fast parsing** | $O(n)$ parse time vs $O(n \log n)$ for nested text formats |
| **Schema-enforced** | Type checking at compile time |

<a id='section-2'></a>
## Section 2: The Protobuf Hierarchy

### Complete Hierarchy Diagram

```
ModelProto
├── ir_version           : int64
├── opset_import         : repeated OpsetIdProto
│   ├── domain           : string
│   └── version          : int64
├── producer_name        : string
├── producer_version     : string
├── domain               : string
├── model_version        : int64
├── doc_string           : string
├── metadata_props       : repeated StringStringEntryProto
│   ├── key              : string
│   └── value            : string
├── graph                : GraphProto
│   ├── name             : string
│   ├── input            : repeated ValueInfoProto
│   │   ├── name         : string
│   │   └── type         : TypeProto
│   │       └── tensor_type : TypeProto.Tensor
│   │           ├── elem_type   : int32 (TensorProto.DataType)
│   │           └── shape       : TensorShapeProto
│   │               └── dim     : repeated Dimension
│   │                   ├── dim_value : int64 (static)
│   │                   └── dim_param : string (symbolic)
│   ├── output           : repeated ValueInfoProto
│   ├── node             : repeated NodeProto
│   │   ├── op_type      : string
│   │   ├── domain       : string
│   │   ├── input        : repeated string
│   │   ├── output       : repeated string
│   │   └── attribute    : repeated AttributeProto
│   │       ├── name     : string
│   │       ├── type     : AttributeType (enum)
│   │       ├── f, i, s  : float, int64, bytes (scalars)
│   │       ├── floats, ints, strings : repeated (lists)
│   │       ├── t        : TensorProto
│   │       └── g        : GraphProto (subgraph!)
│   ├── initializer      : repeated TensorProto
│   │   ├── name         : string
│   │   ├── data_type    : int32
│   │   ├── dims         : repeated int64
│   │   └── raw_data     : bytes
│   └── value_info       : repeated ValueInfoProto (intermediate shapes)
└── functions            : repeated FunctionProto
    ├── name             : string
    ├── domain           : string
    ├── input            : repeated string
    ├── output           : repeated string
    └── node             : repeated NodeProto
```

### Hierarchy Depth

The deepest path is:

$$\text{Model} \to \text{Graph} \to \text{Node} \to \text{Attribute} \to \text{Graph (subgraph)} \to \text{Node} \to \ldots$$

This recursive structure (`GraphProto` inside `AttributeProto` inside `NodeProto` inside `GraphProto`) enables control flow operators.

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install onnx onnxruntime numpy

<a id='section-3'></a>
## Section 3: ModelProto

### Fields and Their Roles

| Field | Type | Required? | Purpose |
|-------|------|:---------:|--------|
| `ir_version` | `int64` | Yes | Identifies the IR spec version used |
| `opset_import` | `[OpsetIdProto]` | Yes | Declares which operator versions are used |
| `graph` | `GraphProto` | Yes | The computation graph |
| `producer_name` | `string` | No | Tool that created the model |
| `producer_version` | `string` | No | Version of the creation tool |
| `domain` | `string` | No | Model's domain namespace |
| `model_version` | `int64` | No | User-defined version number |
| `doc_string` | `string` | No | Human-readable description |
| `metadata_props` | `[StringStringEntry]` | No | Key-value metadata pairs |
| `functions` | `[FunctionProto]` | No | Reusable operator functions |

In [ ]:
import numpy as np
import onnx
from onnx import TensorProto, helper, numpy_helper
from onnx.checker import check_model
from onnx.shape_inference import infer_shapes

# Build a complete model with all ModelProto fields populated
W = numpy_helper.from_array(
    np.array([[1.0, 0.5], [0.5, 1.0]], dtype=np.float32), name='W')
b = numpy_helper.from_array(
    np.array([0.1, -0.1], dtype=np.float32), name='b')

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, ['N', 2])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, ['N', 2])

nodes = [
    helper.make_node('MatMul', ['X', 'W'], ['XW'], name='matmul'),
    helper.make_node('Add', ['XW', 'b'], ['Y'], name='add_bias'),
]

graph = helper.make_graph(nodes, 'ir_demo', [X], [Y], [W, b])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 18)])

model.ir_version = 9
model.producer_name = 'ONNX Tutorial'
model.producer_version = '1.0'
model.domain = 'com.example.tutorial'
model.model_version = 1
model.doc_string = 'Simple linear transformation Y = XW + b'
helper.set_model_props(model, {
    'author': 'Deep Dive Tutorial',
    'license': 'MIT',
    'created': '2024-01-01'})

check_model(model)

print('ModelProto Inspector')
print('=' * 60)
print(f'  ir_version:       {model.ir_version}')
print(f'  producer_name:    {model.producer_name}')
print(f'  producer_version: {model.producer_version}')
print(f'  domain:           {model.domain}')
print(f'  model_version:    {model.model_version}')
print(f'  doc_string:       {model.doc_string}')
print(f'  opset_imports:    {[(o.domain or "ai.onnx", o.version) for o in model.opset_import]}')
print(f'  metadata_props:   {dict((p.key, p.value) for p in model.metadata_props)}')
print(f'  functions:        {len(model.functions)}')

<a id='section-4'></a>
## Section 4: GraphProto

### The Core Container

`GraphProto` is the heart of the model. It contains all the computation.

### Graph Invariants

A valid `GraphProto` must satisfy:

1. **Unique producer rule**: $\forall t \in \text{tensor names}: |\{v : t \in \text{outputs}(v)\}| = 1$
2. **Defined before use**: Every tensor consumed by a node must be either a graph input, an initializer, or produced by a preceding node
3. **Acyclicity**: The induced dependency graph is a DAG
4. **Output coverage**: Every graph output must be produced by some node or be an input/initializer

### Input Resolution Rules

When a tensor name appears in both `graph.input` and `graph.initializer`:

```
Resolution Priority:
  1. If feed_dict[name] is provided at runtime → use feed_dict value
  2. Else if name is in initializer         → use initializer data
  3. Else                                    → ERROR (missing input)
```

In [ ]:
# Inspect the GraphProto
g = model.graph

print('GraphProto Inspector')
print('=' * 60)
print(f'  name:        {g.name}')
print(f'  inputs:      {len(g.input)}')
for inp in g.input:
    t = inp.type.tensor_type
    dims = [d.dim_param or d.dim_value for d in t.shape.dim]
    dtype = TensorProto.DataType.Name(t.elem_type)
    print(f'    {inp.name}: {dtype}{dims}')

print(f'  outputs:     {len(g.output)}')
for out in g.output:
    t = out.type.tensor_type
    dims = [d.dim_param or d.dim_value for d in t.shape.dim]
    dtype = TensorProto.DataType.Name(t.elem_type)
    print(f'    {out.name}: {dtype}{dims}')

print(f'  nodes:       {len(g.node)}')
print(f'  initializers: {len(g.initializer)}')

# Show the complete graph structure
print(f'\nFull Graph Structure:')
print(f'  Inputs: {[i.name for i in g.input]}')
for i, n in enumerate(g.node):
    print(f'  Node[{i}]: {n.op_type}({list(n.input)}) → {list(n.output)}')
print(f'  Outputs: {[o.name for o in g.output]}')

<a id='section-5'></a>
## Section 5: NodeProto and AttributeProto

### AttributeProto Type System

| Type Code | Name | Stored In | Example |
|:---------:|------|-----------|--------|
| 1 | `FLOAT` | `.f` | `alpha=0.01` |
| 2 | `INT` | `.i` | `axis=1` |
| 3 | `STRING` | `.s` | `mode=b"linear"` |
| 4 | `TENSOR` | `.t` | Constant value |
| 5 | `GRAPH` | `.g` | Subgraph (If/Loop body) |
| 6 | `FLOATS` | `.floats` | `[0.1, 0.2, 0.3]` |
| 7 | `INTS` | `.ints` | `perm=[1, 0]` |
| 8 | `STRINGS` | `.strings` | String list |

### Attribute vs Input Decision Criteria

$$\text{Is the value known at graph construction?} \begin{cases} \text{Yes} \to \text{Attribute} \\ \text{No} \to \text{Input (edge)} \end{cases}$$

### Why Subgraph Attributes are Recursive

The `GRAPH` attribute type creates a recursive structure:

```
GraphProto
  └── NodeProto (op_type = "If")
        ├── AttributeProto (name = "then_branch")
        │     └── GraphProto  ← recursive!
        │           └── NodeProto ...
        └── AttributeProto (name = "else_branch")
              └── GraphProto  ← recursive!
                    └── NodeProto ...
```

In [ ]:
# Build a model with various attribute types
X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [2, 3])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, None)

nodes = [
    helper.make_node('Transpose', ['X'], ['Xt'], perm=[1, 0]),
    helper.make_node('Flatten', ['Xt'], ['Y'], axis=1),
]

graph = helper.make_graph(nodes, 'attr_demo', [X], [Y])
model_attr = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 18)])
check_model(model_attr)

# Inspect attributes
print('Attribute Types in Model')
print('=' * 60)
for node in model_attr.graph.node:
    print(f'\n  {node.op_type}:')
    for attr in node.attribute:
        from onnx import AttributeProto as AP
        type_name = AP.AttributeType.Name(attr.type)
        if attr.type == AP.INTS:
            val = list(attr.ints)
        elif attr.type == AP.INT:
            val = attr.i
        elif attr.type == AP.FLOAT:
            val = attr.f
        elif attr.type == AP.STRING:
            val = attr.s.decode()
        else:
            val = '(complex)'
        print(f'    {attr.name}: type={type_name}, value={val}')

<a id='section-6'></a>
## Section 6: TensorProto and ValueInfoProto

### TensorProto — Data Storage

`TensorProto` stores actual tensor data (weights, biases, constants).

| Field | Type | Purpose |
|-------|------|--------|
| `name` | `string` | Tensor identifier |
| `data_type` | `int32` | Element type code |
| `dims` | `[int64]` | Shape dimensions |
| `raw_data` | `bytes` | Dense binary data (preferred) |
| `float_data` | `[float]` | Float values (alternative) |
| `int32_data` | `[int32]` | Int32 values (alternative) |
| `int64_data` | `[int64]` | Int64 values (alternative) |

### Storage Size Formula

$$S = \left(\prod_{i=1}^{r} d_i\right) \cdot \text{sizeof}(\text{dtype})$$

For a weight matrix $W \in \mathbb{R}^{1024 \times 768}$ stored as float32:

$$S = 1024 \times 768 \times 4 = 3{,}145{,}728 \text{ bytes} \approx 3 \text{ MB}$$

### ValueInfoProto — Type Descriptors

`ValueInfoProto` describes the **type and shape** of a tensor without storing its data:

```
ValueInfoProto
├── name    : string
└── type    : TypeProto
    └── tensor_type : TypeProto.Tensor
        ├── elem_type : int32
        └── shape     : TensorShapeProto
            └── dim   : repeated Dimension
                ├── dim_value : int64   (static)
                └── dim_param : string  (symbolic/dynamic)
```

In [ ]:
# TensorProto storage formats
print('TensorProto Storage Comparison')
print('=' * 60)

arr = np.random.randn(3, 4).astype(np.float32)
t = numpy_helper.from_array(arr, name='example')

print(f'  name:      {t.name}')
print(f'  data_type: {TensorProto.DataType.Name(t.data_type)}')
print(f'  dims:      {list(t.dims)}')
print(f'  raw_data:  {len(t.raw_data)} bytes')
print(f'  Expected:  {3 * 4 * 4} bytes (3×4×sizeof(float32))')

roundtrip = numpy_helper.to_array(t)
print(f'  Roundtrip match: {np.allclose(arr, roundtrip)}')

# Size formula for different shapes and dtypes
print(f'\nStorage Size Examples:')
configs = [
    ((768, 768), np.float32, 'BERT attention weight'),
    ((50257, 768), np.float32, 'GPT-2 embedding'),
    ((768, 768), np.float16, 'BERT weight (fp16)'),
    ((768, 768), np.int8, 'BERT weight (quantized)'),
]
for shape, dtype, desc in configs:
    size_bytes = np.prod(shape) * np.dtype(dtype).itemsize
    size_mb = size_bytes / (1024 * 1024)
    print(f'  {desc:30s} {str(shape):15s} {dtype.__name__:8s} → {size_mb:8.2f} MB')

<a id='section-7'></a>
## Section 7: OpSetImportProto and IR Versioning

### The Version Compatibility System

ONNX manages two independent version numbers:

| Version | What it Controls | Set By |
|---------|-----------------|--------|
| **IR Version** | Structure of proto messages | ONNX spec |
| **OpSet Version** | Operator behavior and schemas | `opset_import` |

### Operator Version Resolution

Given opset version $v$, the schema used for operator $\text{op}$ is:

$$\text{schema}(\text{op}, v) = \text{schema}(\text{op}, \max\{s \mid s \leq v, \text{op defined at version } s\})$$

Example: if `Relu` was defined at versions 1, 6, and 14, then under opset 18:

$$\text{schema}(\text{Relu}, 18) = \text{schema}(\text{Relu}, 14)$$

### Multiple Domains

A model can import opsets from different domains:

```python
opset_imports = [
    make_opsetid('', 18),          # standard operators
    make_opsetid('ai.onnx.ml', 3), # ML operators
    make_opsetid('com.myco', 1),   # custom operators
]
```

In [ ]:
# Demonstrate opset versioning
from onnx import defs

ops_to_check = ['MatMul', 'Relu', 'Add', 'Conv', 'BatchNormalization']

print('Operator Version History')
print('=' * 60)
for op in ops_to_check:
    try:
        schema = defs.get_schema(op)
        print(f'  {op:25s} since_version={schema.since_version}')
    except Exception:
        print(f'  {op:25s} not found')

# Test version resolution
print(f'\nVersion Resolution Examples:')
for opset_v in [10, 13, 15, 18]:
    try:
        schema = defs.get_schema('Relu', opset_v)
        print(f'  Relu at opset {opset_v} → uses schema from version {schema.since_version}')
    except Exception as e:
        print(f'  Relu at opset {opset_v} → error: {e}')

<a id='section-8'></a>
## Section 8: Serialization and Binary Format

### How ONNX Models Are Stored

The `.onnx` file is a protobuf-serialized `ModelProto` message:

```
ModelProto (in memory)        .onnx file (on disk)
┌─────────────────┐          ┌──────────────────────┐
│ ir_version=9    │          │ 08 09                │ ← field 1, varint 9
│ graph:          │  ──▶     │ 3A nn nn ...         │ ← field 7, length-delimited
│   node: ...     │ Serialize│   ...graph bytes...  │
│   init: ...     │          │ 42 nn nn ...         │ ← field 8, opset_import
│ opset_import:   │          │   ...opset bytes...  │
│   version=18    │          └──────────────────────┘
└─────────────────┘
```

### Protobuf Wire Types

| Wire Type | Name | Used For |
|:---------:|------|----------|
| 0 | Varint | int32, int64, bool |
| 1 | 64-bit | double, fixed64 |
| 2 | Length-delimited | string, bytes, nested messages |
| 5 | 32-bit | float, fixed32 |

### Size Breakdown Formula

$$S_{\text{model}} = S_{\text{metadata}} + S_{\text{graph\_structure}} + S_{\text{initializers}}$$

In practice, $S_{\text{initializers}}$ dominates for models with learned weights. For a ResNet-50:

$$S_{\text{total}} \approx 97{,}000 \text{ bytes (structure)} + 97{,}500{,}000 \text{ bytes (weights)} \approx 97.5 \text{ MB}$$

In [ ]:
# Analyze model serialization size
import os

serialized = model.SerializeToString()

# Save and compare
path = '/tmp/ir_demo.onnx'
with open(path, 'wb') as f:
    f.write(serialized)

print('Serialization Analysis')
print('=' * 60)
print(f'  Total size:     {len(serialized)} bytes')
print(f'  File size:      {os.path.getsize(path)} bytes')

# Calculate size breakdown
init_size = sum(len(i.raw_data or b'') + sum(len(str(x)) for x in i.float_data)
                for i in model.graph.initializer)
total = len(serialized)
struct = total - init_size
print(f'  Initializers:   ~{init_size} bytes ({100*init_size/total:.1f}%)')
print(f'  Structure:      ~{struct} bytes ({100*struct/total:.1f}%)')

# First few bytes of the protobuf encoding
print(f'\n  First 20 bytes (hex): {serialized[:20].hex()}')

# Roundtrip verification
model2 = onnx.load(path)
assert model2.SerializeToString() == serialized
print(f'  Roundtrip OK: {True}')

os.remove(path)

<a id='section-9'></a>
## Section 9: Inspecting Real Models — Full Hierarchy Walk

In [ ]:
def full_inspection(model):
    """Walk every layer of the ONNX protobuf hierarchy."""
    print('FULL MODEL HIERARCHY INSPECTION')
    print('═' * 60)

    # Level 1: ModelProto
    print(f'\n┌─ ModelProto')
    print(f'│  ir_version:       {model.ir_version}')
    print(f'│  producer_name:    {model.producer_name}')
    print(f'│  producer_version: {model.producer_version}')
    print(f'│  domain:           {model.domain}')
    print(f'│  model_version:    {model.model_version}')
    print(f'│  doc_string:       {model.doc_string[:60]}...' if len(model.doc_string) > 60 else f'│  doc_string:       {model.doc_string}')

    # Level 1.5: OpsetImport
    print(f'│  ┌─ opset_import ({len(model.opset_import)})')
    for o in model.opset_import:
        print(f'│  │  domain={o.domain or "ai.onnx"}, version={o.version}')
    print(f'│  └─')

    # Level 1.5: Metadata
    if model.metadata_props:
        print(f'│  ┌─ metadata_props ({len(model.metadata_props)})')
        for p in model.metadata_props:
            print(f'│  │  {p.key} = {p.value}')
        print(f'│  └─')

    # Level 2: GraphProto
    g = model.graph
    print(f'│  ┌─ GraphProto: "{g.name}"')

    # Level 3: Inputs
    print(f'│  │  ┌─ inputs ({len(g.input)})')
    for inp in g.input:
        t = inp.type.tensor_type
        dims = [d.dim_param or d.dim_value for d in t.shape.dim]
        dtype = TensorProto.DataType.Name(t.elem_type)
        print(f'│  │  │  {inp.name}: {dtype}{dims}')
    print(f'│  │  └─')

    # Level 3: Initializers
    print(f'│  │  ┌─ initializers ({len(g.initializer)})')
    for init in g.initializer:
        dtype = TensorProto.DataType.Name(init.data_type)
        size = len(init.raw_data)
        print(f'│  │  │  {init.name}: {dtype}{list(init.dims)}, {size} bytes')
    print(f'│  │  └─')

    # Level 3: Nodes
    print(f'│  │  ┌─ nodes ({len(g.node)})')
    for n in g.node:
        attrs = ', '.join(f'{a.name}={list(a.ints) or a.i or a.f}'
                          for a in n.attribute) or 'none'
        print(f'│  │  │  {n.op_type}({list(n.input)}) → {list(n.output)}  attrs=[{attrs}]')
    print(f'│  │  └─')

    # Level 3: Outputs
    print(f'│  │  ┌─ outputs ({len(g.output)})')
    for out in g.output:
        t = out.type.tensor_type
        dims = [d.dim_param or d.dim_value for d in t.shape.dim]
        dtype = TensorProto.DataType.Name(t.elem_type)
        print(f'│  │  │  {out.name}: {dtype}{dims}')
    print(f'│  │  └─')

    print(f'│  └─ GraphProto')
    print(f'└─ ModelProto')

full_inspection(model)

In [ ]:
# Build a more complex model to inspect
import onnxruntime as ort

rng = np.random.default_rng(42)
W1 = numpy_helper.from_array(rng.standard_normal((4, 8)).astype(np.float32), name='W1')
b1 = numpy_helper.from_array(rng.standard_normal((8,)).astype(np.float32), name='b1')
W2 = numpy_helper.from_array(rng.standard_normal((8, 2)).astype(np.float32), name='W2')
b2 = numpy_helper.from_array(rng.standard_normal((2,)).astype(np.float32), name='b2')

X = helper.make_tensor_value_info('X', TensorProto.FLOAT, ['batch', 4])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, ['batch', 2])

complex_nodes = [
    helper.make_node('MatMul', ['X', 'W1'], ['h1'], name='fc1'),
    helper.make_node('Add', ['h1', 'b1'], ['h1b'], name='bias1'),
    helper.make_node('Relu', ['h1b'], ['a1'], name='relu1'),
    helper.make_node('MatMul', ['a1', 'W2'], ['h2'], name='fc2'),
    helper.make_node('Add', ['h2', 'b2'], ['Y'], name='bias2'),
]

complex_graph = helper.make_graph(
    complex_nodes, 'mlp_2layer', [X], [Y], [W1, b1, W2, b2])
complex_model = helper.make_model(
    complex_graph, opset_imports=[helper.make_opsetid('', 18)])
complex_model.producer_name = 'IR Deep Dive'
complex_model.doc_string = 'Two-layer MLP: Y = W2 * relu(W1 * X + b1) + b2'
check_model(complex_model)

full_inspection(complex_model)

# Verify it runs
sess = ort.InferenceSession(complex_model.SerializeToString(),
                            providers=['CPUExecutionProvider'])
x = rng.standard_normal((3, 4)).astype(np.float32)
result = sess.run(None, {'X': x})[0]
print(f'\nInference result shape: {result.shape}')
print(f'Values: {result.round(3)}')

<a id='section-10'></a>
## Section 10: Key Takeaways & Interview Questions

### Summary

| Concept | Key Point |
|---------|----------|
| **IR Specification** | Defines the complete structure and semantics of `.onnx` files |
| **Protobuf hierarchy** | `ModelProto → GraphProto → NodeProto → AttributeProto` |
| **Recursive structure** | `GraphProto` can nest inside `AttributeProto` for control flow |
| **TensorProto** | Stores actual data with `raw_data` (preferred) or typed arrays |
| **ValueInfoProto** | Type + shape descriptor without data |
| **Versioning** | IR version (structure) and opset version (operator behavior) are independent |
| **Serialization** | Compact protobuf binary format, weight-dominated size |

### Interview Questions

1. **Q**: What is the difference between `TensorProto` and `ValueInfoProto`?
   - **A**: `TensorProto` stores actual tensor **data** (values, used in initializers). `ValueInfoProto` describes the **type and shape** of a tensor without data (used for graph inputs and outputs). Think of `ValueInfoProto` as the declaration and `TensorProto` as the definition.

2. **Q**: How does the recursive protobuf structure enable control flow?
   - **A**: `AttributeProto` can contain a `GraphProto` (type=GRAPH), which in turn contains `NodeProto`s with their own `AttributeProto`s. This enables operators like `If` to carry `then_branch` and `else_branch` subgraphs as attributes.

3. **Q**: Why does ONNX use two independent version numbers?
   - **A**: The IR version controls the **structural format** (which proto fields exist), while the opset version controls **operator semantics** (how operators compute). Separating them allows operators to evolve without changing the file format, and vice versa.

4. **Q**: For a large model like GPT-2 (774M parameters), what dominates the `.onnx` file size?
   - **A**: Initializer data dominates. With 774M float32 parameters: $774 \times 10^6 \times 4 \approx 3.1$ GB. The graph structure (node definitions, metadata) is typically < 1 MB.

---

**Next:** [Type System and Shapes](../04_Type_System_and_Shapes/) — Type constraints and shape inference.